# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad-imran2891/week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.

One row = one content item, for one client, on one day (report_date + client_hash_id + content_hash_id). Time window: daily facts run from 2025-01-27 to 2026-06-30 (the freshest 3 days before the 2026-07-03 export were deliberately cut).*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: impressions, clicks, position, CTR, sessions, scroll rate — all observed signals, safe to use.
Label/proxy: I'll define decline as a drop in impressions/clicks over a future window (not yet fixed — pending Week 5 modeling).
Context: client_hash_id, content_hash_id — join keys only, not features.
Excluded: health_score, priority_score, action_type — these are product decision outputs, not in this dataset, and would be leakage if reconstructed and fed back in as a feature.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

con.sql(f"SELECT COUNT(*) AS n_clients FROM read_parquet('{REL}/dim_clients.parquet')").show()

con.sql(f"SELECT MIN(report_date), MAX(report_date) FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')").show()

con.sql(f"""
SELECT
  COUNT(*) FILTER (WHERE gsc_impressions IS NOT NULL) AS with_impr,
  COUNT(*) AS total
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()

┌───────────┐
│ n_clients │
│   int64   │
├───────────┤
│       104 │
└───────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬──────────────────┐
│ min(report_date) │ max(report_date) │
│       date       │       date       │
├──────────────────┼──────────────────┤
│ 2025-01-27       │ 2026-06-30       │
└──────────────────┴──────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬─────────┐
│ with_impr │  total  │
│   int64   │  int64  │
├───────────┼─────────┤
│   9841378 │ 9841378 │
└───────────┴─────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This is an unbalanced panel: not every client has full history, and rows before a client's gsc_data_start/ga4_data_start reflect "no tracking yet," not "no traffic." From the dim_clients preview, several clients show null for gsc_data_start or ga4_data_start entirely (e.g. no_search_or_analytics_access clients). Any decline/opportunity label built from this data must define its feature and target windows to never overlap, and should confirm tracking start dates per client before assuming a metric is legitimately zero.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.